# Error Analysis and Anomaly Detection

The objective of this notebook is to investigate fraud cases missed by the final models and understand why they remain difficult to detect.

Three approaches are explored:

1. Investigation of missed fraud cases
2. Nearest-neighbor analysis
3. Isolation Forest anomaly detection

The goal is to determine whether the remaining false negatives exhibit characteristics that distinguish them from detected frauds.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import IsolationForest

In [ ]:
df = pd.read_csv("../data/raw/DataSet.csv")

## Missed Fraud Cases

The final Neural Network model detected 14 out of 16 fraud cases.

The remaining missed frauds are investigated below.

In [ ]:
missed_frauds = [9031, 9043]

df.loc[
    missed_frauds,
    [
        "F115",
        "F2582",
        "F2956",
        "F531",
        "F2737",
        "F3891"
    ]
]

## Nearest-Neighbor Analysis

Nearest-neighbor analysis is used to determine whether the missed frauds resemble legitimate accounts.

If the closest accounts are predominantly non-fraudulent, this suggests that the missed fraud occupies a region of feature space dominated by legitimate behavior.

In [ ]:
df["F115_high"] = (df["F115"] > 0.78).astype(int)

df["F2582_hot"] = (
    (df["F2582"] > -0.03)
    & (df["F2582"] <= 0)
).astype(int)

df["F2956_hot"] = (
    (df["F2956"] > 19)
    & (df["F2956"] <= 32)
).astype(int)

df["F531_hot"] = (
    (df["F531"] > 0.95)
    & (df["F531"] <= 1.35)
).astype(int)

df["F2737_safe"] = (
    (df["F2737"] > 0)
    & (df["F2737"] <= 0.04)
).astype(int)

df["F2582_F531"] = (
    df["F2582_hot"]
    & df["F531_hot"]
).astype(int)

df["fraud_cluster_1"] = (
    df["F2582_hot"]
    & df["F2956_hot"]
    & df["F531_hot"]
).astype(int)

df["f3836_hot"] = (
    (df["F3836"] > 148.596 ) &
    (df["F3836"] <= 20077.212)
).astype(int)

df["F2956_F115"] = (
    df["F2956_hot"] &
    df["F115_high"]
).astype(int)

df["F2582_pos_F2956_low"] = (
    (df["F2582"] > 0.15) &
    (df["F2956"] < 60)
).astype(int)

In [ ]:
features = [
    "F115",
    "F2582",
    "F2956",
    "F531",
    "F2737",
    "F670",
    "F673",
    "F3891",
    "F3889",

    # engineered features

    "F115_high",
    "F2582_hot",
    "F2956_hot",
    "F531_hot",
    "F2737_safe",
    "F2582_F531",
    "fraud_cluster_1",
    "f3836_hot",
    "F2956_F115",
    "F2582_pos_F2956_low"

]

family_cols = [
    "F664","F665","F666",
    "F667","F668","F669",
    "F670","F671","F672",
    "F673","F674","F675",

    "F1","F2","F3","F4","F5","F6","F7","F8","F9","F10","F12"
]

for col in family_cols:
    df[f"{col}_missing"] = df[col].isna().astype(int)

features += [f"{col}_missing" for col in family_cols]

In [ ]:
X = pd.get_dummies(
    df[features],
    columns=["F3891", "F3889"],
    dummy_na=True
)

y = df["F3924"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_train_nn = X_train.fillna(-999)
X_test_nn = X_test.fillna(-999)

nn_search = NearestNeighbors(
    n_neighbors=10
)

nn_search.fit(X_train_nn)

In [ ]:
fraud_id = 9031

idx = X_test.index.get_loc(fraud_id)

distances, neighbors = nn_search.kneighbors(
    X_test_nn.iloc[idx].values.reshape(1, -1)
)

neighbor_ids = X_train.index[neighbors[0]]

pd.DataFrame({
    "account": neighbor_ids,
    "fraud": y_train.loc[neighbor_ids].values
})

In [ ]:
fraud_id = 9043

idx = X_test.index.get_loc(fraud_id)

distances, neighbors = nn_search.kneighbors(
    X_test_nn.iloc[idx].values.reshape(1, -1)
)

neighbor_ids = X_train.index[neighbors[0]]

pd.DataFrame({
    "account": neighbor_ids,
    "fraud": y_train.loc[neighbor_ids].values
})

### Observation

For both missed fraud cases, the nearest neighboring accounts are overwhelmingly legitimate.

This suggests that the missed frauds are not isolated outliers and instead closely resemble normal customer behavior.

## Isolation Forest Experiment

An Isolation Forest model is trained to determine whether the remaining frauds can be identified as anomalous observations.

Unlike supervised models, Isolation Forest does not use fraud labels during training and instead identifies observations that appear statistically unusual.

In [ ]:
X_train_if = X_train.fillna(-999)
X_test_if = X_test.fillna(-999)

iso = IsolationForest(
    n_estimators=500,
    contamination=0.01,
    random_state=42
)

iso.fit(X_train_if)

scores = -iso.score_samples(X_test_if)

results_if = pd.DataFrame({
    "actual": y_test,
    "anomaly_score": scores
}, index=X_test.index)

In [ ]:
results_if.loc[
    [9031, 9043]
].sort_values(
    "anomaly_score",
    ascending=False
)

### Observation

The remaining missed frauds were not assigned exceptionally high anomaly scores.

This indicates that these accounts do not appear statistically unusual relative to the overall customer population.

The result is consistent with the nearest-neighbor investigation.

# Conclusions

The remaining missed fraud cases proved difficult to detect across multiple approaches.

Key findings:

- Neural Networks successfully detected 14 out of 16 fraud cases.
- The remaining frauds closely resembled legitimate accounts.
- Nearest-neighbor analysis showed that the missed frauds were surrounded by non-fraudulent accounts.
- Isolation Forest failed to identify the remaining frauds as strong anomalies.

These findings suggest that the final false negatives may depend on information not present within the available feature set.